Final workable folder and its files view ,  tree view

In [4]:
from googleapiclient.discovery import build
from google.colab import auth
from google.auth import default
from google.colab import drive
import os

# Mount Drive
drive.mount('/content/drive')

# Authentication
auth.authenticate_user()
creds, _ = default()
service = build('drive', 'v3', credentials=creds)

# Output file
output_file = "/content/drive/MyDrive/drive_tree_view.txt"

# Fetch ALL files and folders (including nested ones)
def list_all_files():
    results = []
    page_token = None
    while True:
        response = service.files().list(
            q="trashed = false",
            spaces='drive',
            fields='nextPageToken, files(id, name, mimeType, parents, size)',
            pageSize=1000,
            pageToken=page_token
        ).execute()
        results.extend(response.get('files', []))
        page_token = response.get('nextPageToken')
        if not page_token:
            break
    return results

print("Fetching all files from Google Drive...")
all_items = list_all_files()
print(f"Total items found: {len(all_items)}")

# Build a dictionary: file_id → item details
item_dict = {item['id']: item for item in all_items}

# Build children map: parent_id → list of children items
children_map = {}
for item in all_items:
    parents = item.get('parents', [])
    for parent_id in parents:
        if parent_id not in children_map:
            children_map[parent_id] = []
        children_map[parent_id].append(item)

# Find root items (those with no parents or parents not in drive)
root_items = [item for item in all_items if not item.get('parents') or item['parents'][0] not in item_dict]

# Sort function
def sort_items(items):
    folders = [i for i in items if i['mimeType'] == 'application/vnd.google-apps.folder']
    files = [i for i in items if i['mimeType'] != 'application/vnd.google-apps.folder']
    folders.sort(key=lambda x: x['name'].lower())
    files.sort(key=lambda x: x['name'].lower())
    return folders + files

# Recursive function to print tree
def print_tree(item, prefix="", file_handle=None, is_last=True):
    name = item['name']
    is_folder = item['mimeType'] == 'application/vnd.google-apps.folder'

    connector = "└── " if is_last else "├── "
    marker = "📁 " if is_folder else "📄 "

    line = f"{prefix}{connector}{marker}{name}"
    if not is_folder and 'size' in item:
        size_kb = int(item['size']) / 1024
        if size_kb < 1024:
            line += f" ({size_kb:.1f} KB)"
        else:
            line += f" ({size_kb/1024:.1f} MB)"
    file_handle.write(line + "\n")

    # Get children
    item_children = children_map.get(item['id'], [])
    sorted_children = sort_items(item_children)

    # Recurse into children
    for i, child in enumerate(sorted_children):
        is_last_child = (i == len(sorted_children) - 1)
        new_prefix = prefix + ("    " if is_last else "│   ")
        print_tree(child, new_prefix, file_handle, is_last_child)

# Write to file
with open(output_file, "w", encoding="utf-8") as f:
    f.write("🌳 GOOGLE DRIVE FULL TREE VIEW (A-Z Sorted)\n")
    f.write("="*60 + "\n\n")

    # Handle multiple roots (My Drive + Shared Drives + others)
    sorted_roots = sort_items(root_items)

    for i, root in enumerate(sorted_roots):
        is_last_root = (i == len(sorted_roots) - 1)
        if root['mimeType'] == 'application/vnd.google-apps.folder':
            if 'My Drive' in root.get('name', '') or root['id'] == 'root':
                f.write("🏠 My Drive\n")
            else:
                f.write(f"📂 {root['name']} (Shared Drive or Root)\n")
        print_tree(root, "", f, is_last_root)

print("✅ Complete Tree View ban gaya!")
print("📁 File saved at:")
print(output_file)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Fetching all files from Google Drive...
Total items found: 102
✅ Complete Tree View ban gaya!
📁 File saved at:
/content/drive/MyDrive/drive_tree_view.txt
